# **Experimentación Principal**

- **TFM:** Evaluación experimental de Recursive Language Models para el análisis automatizado de literatura científica

- **Autor:** Juan Antonio Jiménez Cobo

---


## Propósito

Este notebook orquesta la ejecución de la experimentación completa del
trabajo. Las características del sistemas son:
- **Tres sistemas experimentales:** *Large Language Model* (LLM) base, *Retrieval-Augmented Generation* (RAG) y *Recursive Language Model* (RLM)
- **Dos modelos de languaje para cada sistema:** GPT-5.4 mini y Gemini 2.5 Flash, usando GPT-5.4 nano y Gemini 2.5 Flash-Lite como modelos para las sub-llamadas recursivas en RLM
- **Cuatro subconjuntos del corpus:** C1 ($\approx$ 210.000 tokens), C2 ($\approx$ 280.000 tokens), C3 ($\approx$ 360.000 tokens) y C4 ($\approx$ 520.000 tokens)
- **40 consultas divididas en tres niveles de complejidad:** extracción factual (16 consultas), comparación (14 consultas) y sínttesis (10 consultas)
- **2 repeticiones por ejecución:** Para una mayor robustez y representatividad de los resultados

La precisión de cada ejecución se evalúa siguiendo la metodología *LLM-as-judge*, empleando GPT-5.4 mini como modelo juez. Adicionalmente, se calcula el coste y latencia por ejecución. Los resultados se guardan en variables JSON almacenadas en el directorio `RESULTS_DIR`, con el formato `{subset}_{system}_{model_tag}_{query_id}_rep{rep}.json`. También se almacenan los logs de las trayectorias de razonamiento de los RLM mediante la clase `RLMLogger` de la biblioteca de código `rlm` de [Zhang et al](https://github.com/alexzhang13/rlm) para su posterior análisis.

El notebook lleva a cabo los siguientes pasos:

1. Monta Google Drive, instala las dependencias desde `requirements.txt` y
   configura las rutas y parámetros del experimento desde `.env`
2. Inicializa los clientes de las APIs de OpenAI y Google Gemini
3. Carga los registros de cada subconjunto y los formatea como una variable de texto plano
4. Carga las 40 consultas con sus respuestas de referencia desde `queries.json`
5. Define las funciones de control de costes y gestión de checkpoints
6. Ejecuta el sistema LLM base, RAG y RLM para cada combinación de subconjunto,
   modelo, consulta y repetición, guardando cada resultado como archivo JSON
   en `RESULTS_DIR`
7. Evalúa cada respuesta mediante el protocolo *LLM-as-judge* con GPT-5.4 mini
8. Genera el CSV con los resultados de la experimentación

---

## 0. Configuración inicial

### 0.1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### 0.2. Instalación de dependencias

In [ ]:
!pip install -r /content/drive/MyDrive/TFM/requirements.txt --quiet
print("✓ Dependencias instaladas")

### 0.3. Configuración de rutas y modelos

> **IMPORTANTE:** Las rutas se cargan desde el archivo `.env`, ubicado en  el directorio base en Google Drive. En caso de no tener `.env` configurado, se usarán los valores por defecto indicados en el código. Consulta el archivo `.env.example` en el repositorio para ver las variables disponibles

In [ ]:
import os
from dotenv import load_dotenv

# Cargar variables de entorno desde .env en Google Drive
load_dotenv('/content/drive/MyDrive/proyect/.env')

# Directorio base del proyecto
BASE_DIR = os.getenv('BASE_DIR', '/content/drive/MyDrive/proyect')

# Directorio con los archivos .json con el texto preprocesado de los registros del corpus
CORPUS_DIR = os.getenv('CORPUS_PROCESSED_DIR', f'{BASE_DIR}/corpus/processed')

# Directorio con los subconjuntos del corpus
SUBSETS_DIR = f'{BASE_DIR}/corpus/subsets'

# Ruta del archivo .json con las consultas y sus respuestas de referencia
QUERIES_PATH = os.getenv('QUERIES_PATH', f'{BASE_DIR}/queries/queries.json')

# Directorio para almacenar los resultados de las ejecuciones del pipeline de experimentación
RESULTS_DIR = os.getenv('RESULTS_DIR',  f'{BASE_DIR}/experiments/results')

# Directorio para almacenar los registros de las trayectorias de razonamiento de los sistemas RLM
RLM_LOGS_DIR = os.getenv('RLM_LOGS_DIR', f'{BASE_DIR}/experiments/rlm_logs')

# Directorio para almacenar el índice vectorial de ChromaDB
CHROMA_DIR = os.getenv('CHROMA_DIR',   f'{BASE_DIR}/experiments/rag/chroma_index')


A continuación se definen los modelos y sus configuraciones empleadas para este experimento. Configúrese según las características de la experimentación a ejecutar

Modelos:
- `GPT_ROOT`: Modelo raíz de la familia GPT (`gpt-5.4-mini`)
- `GPT_SUB`: Modelo para las sub-llamadas recursivas del sistema RLM de la familia GPT (`gpt-5.4-nano-2026-03-17`)
- `GEMINI_ROOT`: Modelo raíz de la familia Gemini (`gemini-2.5-flash`)
- `GEMINI_SUB`: Modelo para las sub-llamadas recursivas del sistema RLM de la familia Gemini (`gemini-2.5-flash-lite`)
- `EMBED_MODEL`: Modelo para generar las representaciones vectoriales en el sistema RAG (`text-embedding-3-small`)
- `JUDGE_MODEL`: Modelo para evaluar las respuestas mediante *LLM-as-judge* (`gpt-5.4-mini`)


Hiperparámetros
- `CHUNK_SIZE`: Tamaño de chunk en RAG
- `CHUNK_OVERLAP`: Tamaño de solapamiento entre chunks consecutivos en RAG
- `TOP_K `: Número de fragmentos recuperados en RAG
- `N_REPS`: Número de repeticiones por ejecución     
- `BUDGET_USD`: Límite de prespuesto de la experimentación en USD (el sistema se detendrá de forma automática cuando el coste acumulado supere el umbral)    
- `MAX_ITERATIONS`: Límite de iteraciones del bucle de razonamiento de RLM
- `GPT_SKIP`: Subconjuntos del corpus que no son procesados por `gpt-5.4-mini` en LLM base (dada la restircción de la ventana de contexto)
- `SUBSETS`: Identificadores de los subconjuntos del corpus

Los precios por millón de tokens en USD de los modelos han sido extraídos de la documentación oficial de cada modelo ([GPT-5.4 mini y  nano](https://openai.com/es-ES/index/introducing-gpt-5-4-mini-and-nano/), [Gemini 2.5 Flash y Flash-Lite](https://ai.google.dev/gemini-api/docs/pricing?hl=es-419), y [text-embedding-3-small](https://developers.openai.com/api/docs/pricing))

In [ ]:
# Modelos
GPT_ROOT    = 'gpt-5.4-mini'
GPT_SUB     = 'gpt-5.4-nano-2026-03-17'
GEMINI_ROOT = 'gemini-2.5-flash'
GEMINI_SUB  = 'gemini-2.5-flash-lite'
EMBED_MODEL = 'text-embedding-3-small'
JUDGE_MODEL = 'gpt-5.4-mini'

# Hiperparámetros
CHUNK_SIZE     = 512
CHUNK_OVERLAP  = 50
TOP_K          = 5
N_REPS         = 2
BUDGET_USD     = 250.0
MAX_ITERATIONS = 15
GPT_SKIP       = ['C3', 'C4']
SUBSETS        = ['C1', 'C2', 'C3', 'C4']

# Precios por millón de tokens (USD)
PRICES = {
    'gpt-5.4-mini':           {'input': 0.75,  'output': 4.50},
    'gpt-5.4-nano':           {'input': 0.20,  'output': 1.25},
    'gemini-2.5-flash':       {'input': 0.30,  'output': 2.50},
    'gemini-2.5-flash-lite':  {'input': 0.10,  'output': 0.40},
    'text-embedding-3-small': {'input': 0.02,  'output': 0.0},
}

## 1. Imports, utilidades y API

Se inicializan los clientes de las API de OpenAI y Google Gemini.

> **IMPORTANTE:**
> - Las claves de acceso se cargan desde el archivo `.env`, ubicado en  el directorio base en Google Drive. Consulta el archivo `.env.example` en el repositorio para ver las variables disponibles.
> - La librería `rlm` únicamente da acceso a modelos de OpenAI y Anthropic. Para emplear modelos de la familia Gemini, se debe usar el endpoint de Google compatible con la interfaz de OpenAI mediante el enlace https://generativelanguage.googleapis.com/v1beta/openai/

Adicionalmente, se configura el tokenizador `cl100k_base` de la biblioteca `tiktoken`, el mismo tokenizador que se emplea en GPT-4, para estimar la longitud en tokens de cada registro procesado.

In [ ]:
import os, json, time, csv, tiktoken
from pathlib import Path
from datetime import datetime

from llama_index.core import VectorStoreIndex, Document, Settings, StorageContext
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.callbacks import CallbackManager, TokenCountingHandler
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI as LlamaOpenAI
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb
from rlm import RLM
from rlm.logger import RLMLogger
from openai import OpenAI as OpenAIClient
from google import genai
import nest_asyncio
nest_asyncio.apply()

# Crear carpetas de salida necesarias
for d in [RESULTS_DIR, RLM_LOGS_DIR, CHROMA_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

# Cargar claves de API desde variables de entorno
openai_key = os.environ['OPENAI_API_KEY']
gemini_key = os.environ['GOOGLE_API_KEY']

# Cliente OpenAI para el sistema LLM base GPT y el juez LLM
openai_client = OpenAIClient(api_key=openai_key)

# Cliente Gemini mediante endpoint compatible con la interfaz OpenAI
gemini_openai_client = OpenAIClient(
    base_url='https://generativelanguage.googleapis.com/v1beta/openai/',
    api_key=gemini_key
)

# Tokenizador cl100k_base para calcular costes del RLM desde los logs JSONL
enc = tiktoken.get_encoding('cl100k_base')


## 2. Funciones auxiliares

### 2.1. Cálculo de costes

- Los costes por ejecución de los sistemas LLM base y RAG se calculan con `calculate_cost()` a partir del número de tokens de entrada y salida que procesa el sistema, multiplicado por el precio por millón de tokens de cada modelo definido en `PRICES`.
- En el caso del sistema RLM, la llamada al modelo no agrega el coste de las trayectorias de razonamiento recursivas. Para ello, se define la función `calculate_rlm_cost_from_log()`, la cual calcula el coste total del RLM a partir de los registros en formato `.jsonl` generados por la clase `RLMLogger`, sumando los tokens procesados por el modelo raíz y por cada sub-llamada de forma independiente
- Adicionalmente, `check_budget()` se define como un checkpoint que permite que el coste total del sistema no supere el límite establecido en `BUDGET_USD`, devolviendo un `RuntimeError` en caso de que el coste agregado supere dicha cantidad.

In [ ]:
total_cost_usd = 0.0

def check_budget():
  '''Checkpoint que devuelve RuntimeError en caso de que el coste total de
  la experimentación supere el límite establecido en la variable BUDGET_USD'''
  if total_cost_usd >= BUDGET_USD:
      raise RuntimeError(f'PRESUPUESTO AGOTADO: ${total_cost_usd:.2f} >= ${BUDGET_USD}')

def calculate_cost(model, input_tokens, output_tokens):
  '''Calcula el coste de la llamada multiplicando el número de tokens de
  entrada y salida por el precio del modelo'''
  if model not in PRICES:
      return 0.0
  p = PRICES[model]
  return (input_tokens * p['input'] + output_tokens * p['output']) / 1_000_000

def calculate_rlm_cost_from_log(jsonl_path, root_model, sub_model):
  '''Calcula el coste del sistema RLM a partir de los registros JSONL generados
  en cada ejecución'''
  root_input = root_output = sub_input = sub_output = 0
  with open(jsonl_path, encoding='utf-8') as f:
      for line in f:
          entry = json.loads(line)
          if entry.get('type') != 'iteration':
              continue
          for msg in entry.get('prompt', []):
              root_input += len(enc.encode(msg.get('content', '')))
          root_output += len(enc.encode(entry.get('response', '')))
          for block in entry.get('code_blocks', []):
              for call in block.get('rlm_calls', []):
                  sub_input  += len(enc.encode(call.get('prompt', '')))
                  sub_output += len(enc.encode(call.get('response', '')))
  cost = calculate_cost(root_model, root_input, root_output) + \
          calculate_cost(sub_model,  sub_input,  sub_output)
  return cost, root_input + sub_input, root_output + sub_output


### 2.2. Checkpoints

El sistema incluye funciones para almacenar los registros y resultados obtenidos y retomar la ejecución en caso de interrupción sin perder ni tener que repetir los resultados ya obtenidos. Los resultados se almacenan en `RESULTS_DIR` con el formato `{subset}_{system}_{model_tag}_{query_id}_rep{rep}.json`.

In [ ]:
def result_path(subset, system, model_tag, query_id, rep):
  '''Construye la ruta del archivo JSON con los resultados de cada ejecución'''
  return os.path.join(RESULTS_DIR, f'{subset}_{system}_{model_tag}_{query_id}_rep{rep}.json')

def save_result(subset, system, model_tag, query_id, rep, data):
  '''Guarda el archivo JSON con los resultados de una ejecución en el directorio
  RESULTS_DIR'''
  with open(result_path(subset, system, model_tag, query_id, rep), 'w', encoding='utf-8') as f:
      json.dump(data, f, ensure_ascii=False, indent=2)

def already_done(subset, system, model_tag, query_id, rep):
  '''Comprueba si el archivo con los resultados de una ejecución ya existe,
  evitando así repetir ejecuciones ya realizadas'''
  return os.path.exists(result_path(subset, system, model_tag, query_id, rep))

## 3. Carga de subconjuntos del corpus

Los 4 subconjuntos del corpus se cargan desde el archivo `SUBSETS_DIR/subsets.json`, el cual contiene los identificadores de los registro incluidos en cada subconjunto.

Adicionalmente, se incluyen dos funciones auxiliares:
- `load_subset()`: carga los archivos JSON preprocesados de los registros
  correspondientes a un subconjunto dado desde `CORPUS_DIR`
- `corpus_to_prompt()`: agrega los registros del subconjunto como texto
  plano para su uso como contexto en el sistema LLM base, precediendo cada
  registro con un encabezado en el formato
  `--- título (autores, año) ---`

In [ ]:
# Carga del archivo subsets.json con los ID de los registros incluidos en cada subconjunto
with open(os.path.join(SUBSETS_DIR, 'subsets.json'), encoding='utf-8') as f:
    SUBSET_DEFINITIONS = json.load(f)

# Imprimir los subconjuntos disponibles y el número de registros en cada uno
print('Subconjuntos disponibles:')
for name, paper_ids in SUBSET_DEFINITIONS.items():
    print(f'  {name}: {len(paper_ids)} papers ({paper_ids[0]} ... {paper_ids[-1]})')

#-------- Funciones auxiliares ---------------------
def load_subset(subset_name):
  '''Carga los archivos JSON de todos los registros de un subconjunto'''
  paper_ids = SUBSET_DEFINITIONS[subset_name]
  papers = []
  for paper_id in paper_ids:
      with open(os.path.join(CORPUS_DIR, f'{paper_id}.json'), encoding='utf-8') as f:
            papers.append(json.load(f))
  return papers

def corpus_to_prompt(papers):
  '''Agrega los textos preprocesados de los registros de un subconjunto como
  texto plano. Cada registro va precedido de un identificador de la forma
  ---título (autores, año).'''
  parts = []
  for p in papers:
      parts.append(f"--- {p.get('title','')} ({p.get('authors','')}, {p.get('year','')}) ---")
      parts.append(p['text'])
      parts.append('')
  return '\n'.join(parts)

## 4. Carga de consultas

Las consultas se cargan desde la ruta `QUERIES_PATH` en formato `.json`

In [ ]:
# Cargar consultas en formato .json
with open(QUERIES_PATH, encoding='utf-8') as f:
    queries_data = json.load(f)
queries = queries_data['queries']

# Imprimir número de consultas cargadas y dividir por nivel de complejidad
print(f'{len(queries)} consultas cargadas')
for level in ['factual', 'comparison', 'synthesis']:
    print(f'  {level}: {sum(1 for q in queries if q["level"] == level)}')

## 5. Sistema LLM base

El sistema LLM base implementa llamadas directas a la API de cada modelo con el corpus completo del subconjunto en el contexto de entrada.

### 5.1. Prompt del sistema

Además del contexto de entrada, el LLM base recibe un prompt de sistema en el que se le instruye mediante *role prompting* a actuar como un asistente experto en el análisis de literatura científica, además de restringir sus respuestas únicamente a la información que recibe en el contexto. Adicionalmente, el modelo recibe una serie de indicaciones sobre cómo tiene que actuar para responder a las consultas de cada nivel de complejidad.

In [ ]:
SYSTEM_PROMPT = """You are an expert assistant in scientific literature analysis.
Answer exclusively based on the documents provided.
Do not add information that is not present in the provided documents.

Follow these response guidelines:
- For questions asking you to list or identify specific elements,
  respond directly with the requested information only.
- For questions asking you to compare approaches or methodologies,
  structure your response addressing each element of the comparison
  and cite the specific papers involved.
- For questions asking you to synthesize findings across papers,
  provide an integrative response that goes beyond describing
  individual papers and identifies patterns or gaps in the literature.

You will first receive the question to answer, followed by the corpus of
scientific papers to use as your knowledge base.
"""

### 5.2. Llamadas al modelo

El sistema LLM base se implementa mediante llamadas directas a la API de cada modelo. Ambos modelos reciben el prompt del sistema, seguido de la consulta correspondiente y del corpus del subconjunto procesado. Se implementan dos funciones independientes para los dos modelos utilizados (GPT-5.4 mini y Gemini 2.5 Flash):

- `run_baseline_gpt()`: realiza la llamada a la API de OpenAI con temperatura 0 (para garantizar reproducibilidad de resultados). Incluye un mecanismo de reintento automático de 3 intentos con una espera de 60 segundos para evitar errores TPM por límite de tasa. En caso de error por límmite de tasa, devuelve el error y, en caso de ejecució exitosa, devuelve los siguientes campos:
    - `response`: Respuesta del sistema
    - `latency_s`: latencia del sistema en segundos
    - `input_tokens`: número de tokens de entrada procesados por el sistema
    - `ouput_tokens`: número de tokens de salida del sistema (tokens de la respuesta)
    - `cost_usd`: coste de la llamada en USD
- `run_baseline_gemini()`: realiza la llamada a la API de Google meidante el cliente nativo `google.genai.Client()`. Se incluye un mecanismo de llamada de 65 segundos para subconjuntos con menos de 400.000 tokens y de 130 segundos para subconjuntos que superen dicho umbral, con el fin de evitar errores de saturación de la API. En caso de ejecución exitosa, se devuelven los mismos campos que `run_baseline_gpt()`.

In [ ]:
def run_baseline_gpt(corpus_text, query, model):
  # Variable global del coste acumulado
  global total_cost_usd

  # Comprobación de si el coste acumulado supera el límite
  check_budget()

  # El sistema incluye 3 reintentos por si se obtiene un error de límite de tasa
  MAX_RETRIES = 3
  for attempt in range(MAX_RETRIES):
      try:
          # Tiempo inicial al comenzar la llamada (usado para calcular la latencia)
          t0 = time.time()
          # Llamada al modelo
          resp = openai_client.chat.completions.create(
              model=model, temperature=0,
              messages=[
                  {'role': 'system', 'content': SYSTEM_PROMPT},
                  {'role': 'user', 'content': f'Question: {query}\n\nContext: {corpus_text}'}
              ]
          )
          # Cálculo de latencia
          latency = time.time() - t0
          # Número de tokens de entrada y salida
          inp, out = resp.usage.prompt_tokens, resp.usage.completion_tokens
          # Cálculo del coste de la llamada
          cost = calculate_cost(model, inp, out)
          total_cost_usd += cost
          return {'response': resp.choices[0].message.content,
                  'latency_s': round(latency, 3),
                  'input_tokens': inp, 'output_tokens': out,
                  'cost_usd': round(cost, 6)}
      except Exception as e:
          # Si se obtiene error de tasa, se esperan 60 segundos hasta el siguiente intento
          if '429' in str(e) and attempt < MAX_RETRIES - 1:
              wait = 60  # esperar 60 segundos para que se renueve el TPM
              print(f'Rate limit TPM. Esperando {wait}s... (intento {attempt+1}/{MAX_RETRIES})')
              time.sleep(wait)
          else:
              raise

google_client = genai.Client(api_key=gemini_key)

def run_baseline_gemini(corpus_text, query, model):
  # Variable global del coste acumulado
  global total_cost_usd

  # Comprobación de si el coste acumulado supera el límite
  check_budget()

  # Prompt completo (Prompt sistema + consulta + contexto)
  prompt = f"{SYSTEM_PROMPT}\n\nQuestion: {query}\n\nContext: {corpus_text}"

  # Tiempo al inicio de la consulta (para calcular latencia)
  t0 = time.time()

  # Llamada al modelo
  response = google_client.models.generate_content(
      model=model,
      contents=prompt
  )
  # Cálculo de latencia
  latency = time.time() - t0

  # Número de tokens de entrada y salida
  inp  = response.usage_metadata.prompt_token_count
  out  = response.usage_metadata.candidates_token_count

  # Coste de la llamada
  cost = calculate_cost(model, inp, out)
  total_cost_usd += cost

  # Calcular espera basada en el tamaño del corpus
  tokens_per_call = inp + out
  if tokens_per_call > 400000:
      wait = 130
  else:
      wait = 65
  time.sleep(wait)

  return {
      'response':      response.text,
      'latency_s':     round(latency, 3),
      'input_tokens':  inp,
      'output_tokens': out,
      'cost_usd':      round(cost, 6)
  }


## 6. Sistema RAG

El sistema RAG se implementa mediante dos funciones:

- `build_rag_index()`: construye el índice vectorial del subconjunto del
corpus. Cada registro se carga como objeto `Document` de LlamaIndex con sus
metadatos (`paper_id`, `title`, `year`). Los documentos se segmentan en
fragmentos de `CHUNK_SIZE=512` tokens con `CHUNK_OVERLAP=50` tokens de
solapamiento mediante `SentenceSplitter`. Los embeddings se generan con el
modelo `text-embedding-3-small` de OpenAI y se almacenan en una colección
ChromaDB con el nombre `{subset}_{model_tag}` en `CHROMA_DIR`.
El coste de generación de embeddings se registra y acumula en `total_cost_usd`.
El modelo generador se configura con temperatura 0, usando `LlamaOpenAI` para
GPT-5.4 mini y `GoogleGenAI` para Gemini 2.5 Flash.

- `run_rag()`: ejecuta una consulta sobre el índice vectorial. Recupera
los `TOP_K=5` fragmentos más similares semánticamente a la consulta mediante
similitud coseno y los envía al modelo generador junto a la consulta. Los
tokens consumidos se registran mediante `TokenCountingHandler`, empleando el encoding de GPT-4o. La función devuelve los mismos campos que `run_baseline_gpt()` y `run_baseline_gemini()`, junto a los metadatos de los fragmentos recuperados (`paper_id` y puntuación de similitud), que permiten verificar qué documentos del corpus contribuyeron a cada respuesta.

In [ ]:
def build_rag_index(papers, model, collection_name):
  # Variable global del coste acumulado
  global total_cost_usd

  # Cada registro del corpus se carga como objeto Document
  docs = [Document(text=p['text'],
                    metadata={'paper_id': p['paper_id'],
                              'title': p.get('title',''),
                              'year': str(p.get('year',''))})
          for p in papers]
  # Contador de tokens
  tc = TokenCountingHandler(tokenizer=tiktoken.encoding_for_model('gpt-4o').encode)
  Settings.callback_manager = CallbackManager([tc])
  # text-embedding-3-small como modelo de embeddings
  Settings.embed_model = OpenAIEmbedding(model=EMBED_MODEL)
  # Los modelos generadores se cargan mediante LlamaIndex (temperatura 0)
  Settings.llm = LlamaOpenAI(model=model, temperature=0) if 'gpt' in model \
              else GoogleGenAI(model=model, temperature=0)
  cc = chromadb.PersistentClient(path=CHROMA_DIR).get_or_create_collection(collection_name)
  # Crear ínidce vectorial a partir de los documentos cargados como Document
  idx = VectorStoreIndex.from_documents(
      docs,
      storage_context=StorageContext.from_defaults(vector_store=ChromaVectorStore(chroma_collection=cc)),
      transformations=[SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)]
  )
  # Calcular coste del modelo de embeddings
  embed_cost = calculate_cost(EMBED_MODEL, tc.total_embedding_token_count, 0)
  total_cost_usd += embed_cost
  return idx, tc, embed_cost

def run_rag(index, query, model, tc):
  # Variable global del coste acumulado
  global total_cost_usd
  # Comprobación de si el coste acumulado supera el límite
  check_budget()
  tc.reset_counts()
  # Tiempo al iniciar la consulta (para calcular la latencia)
  t0 = time.time()
  # Respuesta del modelo generador recuperando los top_k=5 fragmentos más similares
  resp = index.as_query_engine(similarity_top_k=TOP_K).query(query)
  # Cálculo de latencia
  latency = time.time() - t0
  # Número de tokens de entrada y salida
  inp, out = tc.prompt_llm_token_count, tc.completion_llm_token_count
  # Coste de la ejecución
  cost = calculate_cost(model, inp, out)
  total_cost_usd += cost
  return {'response': resp.response,
          'latency_s': round(latency, 3),
          'input_tokens': inp, 'output_tokens': out,
          'cost_usd': round(cost, 6),
          'chunks': [{'paper_id': n.metadata.get('paper_id'),
                      'score': round(n.score, 4)} for n in resp.source_nodes]}


## 7. Sistema RLM

Se definen dos funciones para implementar el sistema RLM, a partir de la biblioteca oficial `rlm` de [Zhang et al.](https://github.com/alexzhang13/rlm):

- `build_rlm()`: instancia el objeto `RLM` con el modelo raíz y el modelo
de sub-llamadas. Para los modelos Gemini, tanto el modelo raíz como el modelo
de sub-llamadas acceden a la API mediante el endpoint compatible con la
interfaz OpenAI proporcionado por Google
(`https://generativelanguage.googleapis.com/v1beta/openai/`), lo que permite
emplear el backend OpenAI de la biblioteca sin modificaciones en el código.
El número máximo de iteraciones del bucle recursivo se limita a
`MAX_ITERATIONS=15`. Las trayectorias de razonamiento se registran
automáticamente en `log_dir` mediante `RLMLogger`.

- `run_rlm()`: ejecuta una consulta sobre el sistema RLM. El corpus del
subconjunto se pasa como contexto mediante `rlm.completion()` junto al prompt
raíz, que incluye como primer paso obligatorio la inspección de los primeros
2.000 caracteres del contexto para forzar la interacción con el entorno REPL.
Una vez completada la ejecución, la función localiza el archivo JSONL generado
por `RLMLogger`, lo renombra con un nombre descriptivo siguiendo el formato
`{subset}_rlm_{model_tag}_{query_id}_rep{rep}.jsonl`, y calcula el coste
estimado mediante `calculate_rlm_cost_from_log()`. La función devuelve la
respuesta final, la latencia, el coste estimado, el número de iteraciones
del bucle recursivo y la ruta al log JSONL de la trayectoria.

In [ ]:
def build_rlm(root_model, sub_model, log_dir):

  # Crear la carpeta para los registros de las trayectorias RLM en caso de no existir
  Path(log_dir).mkdir(parents=True, exist_ok=True)

  # Los modelos Gemini acceden a la API mediante el endpoint compatible con la
  # interfaz OpenAI
  if 'gemini' in root_model:
      backend_kwargs = {
          'model_name': root_model,
          'base_url': 'https://generativelanguage.googleapis.com/v1beta/openai/',
          'api_key': gemini_key
      }
      sub_backend_kwargs = {
          'model_name': sub_model,
          'base_url': 'https://generativelanguage.googleapis.com/v1beta/openai/',
          'api_key': gemini_key
      }
  else:
      backend_kwargs = {'model_name': root_model}
      sub_backend_kwargs = {'model_name': sub_model}

  return RLM(
      backend='openai',
      backend_kwargs=backend_kwargs,
      other_backends=['openai'],
      other_backend_kwargs=[sub_backend_kwargs],
      max_iterations=MAX_ITERATIONS,
      verbose=False,
      logger=RLMLogger(log_dir=log_dir)
  )


def run_rlm(rlm, corpus_text, query, root_model, sub_model, log_dir, query_id, rep):

  # Variable global del coste acumulado
  global total_cost_usd
  # Comprobación de si el coste acumulado supera el límite
  check_budget()

  # Prompt raíz (accesible por el modelo raíz). Fuerza a inspeccionar los primeros
  # 2.000 caracteres de contexto
  root_prompt = f"""Answer the following question based exclusively on the scientific papers corpus available in the context variable.

  MANDATORY FIRST STEP: You MUST start by inspecting the context. Execute this exact code block:

  ```repl
  print(context[:2000])
  ```

  Response guidelines:
    - For questions asking you to list or identify specific elements, respond directly with the requested information only.
    - For questions asking you to compare approaches or methodologies, structure your response addressing each element of the comparison and cite the specific papers involved.
    - For questions asking you to synthesize findings across papers, provide an integrative response that identifies patterns or gaps in the literature.

  QUESTION: {query}
  """
  t0 = time.time()
  # Llamada al RLM
  completion = rlm.completion(
      prompt=corpus_text,
      root_prompt=root_prompt
  )
  # Cálculo de latencia
  latency = time.time() - t0

  # Calcular coste desde el log JSONL generado por RLMLogger
  log_files = sorted(Path(log_dir).glob('*.jsonl'), key=os.path.getmtime)
  cost = inp = out = 0
  n_iterations = 0
  if log_files:
      latest_log = str(log_files[-1])

      # Renombrar archivo del log del RLM con nombre descriptivo
      descriptive_path = os.path.join(
          log_dir,
          f'{subset}_rlm_{tag}_{query_id}_rep{rep}.jsonl'
      )
      os.rename(latest_log, descriptive_path)
      latest_log = descriptive_path

      cost, inp, out = calculate_rlm_cost_from_log(latest_log, root_model, sub_model)
      # Contar iteraciones
      with open(latest_log, encoding='utf-8') as f:
          n_iterations = sum(1 for line in f
                              if json.loads(line).get('type') == 'iteration')

  total_cost_usd += cost
  return {'response': completion.response,
          'latency_s': round(latency, 3),
          'input_tokens': inp, 'output_tokens': out,
          'cost_usd': round(cost, 6),
          'n_iterations': n_iterations,
          'log_file': str(log_files[-1]) if log_files else None}


## 8. LLM-as-judge

### 8.1. Prompts del modelo

Se definen dos prompts de evaluación distintos según el tipo de consulta:

- `JUDGE_PROMPT_FACTUAL`: evalúa consultas de extracción factual. Los
criterios de evaluación priorizan la corrección conceptual sobre la coincidencia
textual exacta, penalizando diferencias menores de capitalización, artículos
u orden de palabras únicamente si afectan al contenido de la respuesta.
También penaliza respuestas con contexto excesivo no solicitado o elementos
clave omitidos.

- `JUDGE_PROMPT_COMPARISON_SYNTHESIS`: evalúa consultas de comparación y
síntesis. A diferencia del prompt factual, la respuesta de referencia se
emplea como guía orientativa y no como coincidencia requerida, admitiendo
respuestas con citas distintas o enfoques alternativos siempre que sean
correctas. Penaliza respuestas factualmente incorrectas o que omitan la comparación o síntesis en la respuesta.

En ambos casos el juez responde exclusivamente en formato JSON con los campos
`score` (entero 1-5) y `justification` (texto).

In [ ]:
JUDGE_PROMPT_FACTUAL = """You are an expert evaluator of scientific literature
question-answering systems.

Your task is to evaluate whether a system response correctly answers a factual
question about scientific papers.

Question: {query}
Reference answer: {ground_truth}
System response: {response}

Evaluation criteria:
- Focus on CONCEPTUAL correctness, not textual matching
- Minor differences in capitalization, articles, word order, or punctuation
  should NOT affect the score
- Penalize responses that add excessive unrequested context instead of
  directly answering the question
- Penalize responses that are incomplete or missing key elements

Score from 1 to 5:
1 - Incorrect or completely irrelevant
2 - Partially correct but missing key elements
3 - Mostly correct but incomplete or with minor errors
4 - Correct and complete, may have trivial wording differences
5 - Fully correct, complete and direct

Example:
Question: List the three main components of system X.
Reference: Components A, B and C.
Response: The system has component A, component B, and component C.
Score: 5 (same content, trivial wording difference)

Respond ONLY with valid JSON: {{"score": <1-5>, "justification": "<brief explanation>"}}
"""

In [ ]:
JUDGE_PROMPT_COMPARISON_SYNTHESIS = """You are an expert evaluator of scientific
literature question-answering systems.

Your task is to evaluate whether a system response correctly answers a question
that requires comparing or synthesizing information from multiple scientific papers.

Question: {query}
Reference answer: {ground_truth}
System response: {response}

Evaluation criteria:
- The reference answer is a GUIDE, not a required exact match
- A response using different valid citations or approaching the answer from
  a different angle should still receive a high score if it is correct
- Evaluate accuracy, completeness of the comparison/synthesis, and relevance
- Additional relevant context beyond the reference is acceptable and may
  indicate deeper understanding
- Penalize responses that are factually incorrect or miss the key
  comparative/synthetic insight

Score from 1 to 5:
1 - Incorrect or does not address the comparison/synthesis requested
2 - Partially addresses the question with significant gaps or errors
3 - Addresses the question but misses important elements of the comparison/synthesis
4 - Correct and well-reasoned, may use different but valid citations
5 - Complete, accurate synthesis/comparison, well-grounded in the literature

Respond ONLY with valid JSON: {{"score": <1-5>, "justification": "<brief explanation>"}}
"""

### 8.2. Ejecución del LLM juez

La evaluación de cada consulta se ejecuta mediante la función `run_judge()`, que selecciona el prompt adecuado según el nivel de la
consulta, llama a la API del modelo juez con temperatura 0 y parsea la respuesta JSON. En
caso de error de parseo, devuelve `score=-1` para identificar el fallo en
el análisis posterior.

In [ ]:
def run_judge(query, ground_truth, response, level):

  # Variable global del coste acumulado
  global total_cost_usd

  # Elección del prompt en función del tipo de consulta
  if level == 'factual':
      prompt = JUDGE_PROMPT_FACTUAL.format(
          query=query, ground_truth=ground_truth, response=response)
  else:
      prompt = JUDGE_PROMPT_COMPARISON_SYNTHESIS.format(
          query=query, ground_truth=ground_truth, response=response)

  # Llamada al modelo juez (GPT-5.4 mini, temperatura=0)
  result = openai_client.chat.completions.create(
      model=JUDGE_MODEL, temperature=0,
      messages=[{'role': 'user', 'content': prompt}]
  )

  # Número de tokens de entrada/salida y coste
  inp, out = result.usage.prompt_tokens, result.usage.completion_tokens
  cost = calculate_cost(JUDGE_MODEL, inp, out)
  total_cost_usd += cost

  # Parseo de la respuesta JSON
  try:
      parsed = json.loads(result.choices[0].message.content)
      return {'score': int(parsed['score']),
              'justification': parsed.get('justification', ''),
              'cost_usd': round(cost, 6)}
  # Si no se puede parsear, se devuelve una puntuación de -1
  except Exception:
      return {'score': -1, 'justification': 'ERROR: parse failed',
              'cost_usd': round(cost, 6)}


## 9. Pipeline principal de experimentación

Bucle principal que orquesta la ejecución completa del experimento iterando
sobre todos los subconjuntos del corpus, sistemas experimentales, modelos de
lenguaje, consultas y repeticiones. Para cada combinación se comprueba si
ya existe un checkpoint en `RESULTS_DIR` mediante `already_done()` y, en caso
afirmativo, la ejecución se omite para evitar reejecutar resultados ya
obtenidos.

Las configuraciones evaluadas son:

| Sistema | Modelo raíz | Modelo sub-llamadas |
|---|---|---|
| LLM base | GPT-5.4 mini | — |
| LLM base | Gemini 2.5 Flash | — |
| RAG | GPT-5.4 mini | — |
| RAG | Gemini 2.5 Flash | — |
| RLM | GPT-5.4 mini | GPT-5.4 nano |
| RLM | Gemini 2.5 Flash | Gemini 2.5 Flash-Lite |

Para cada ejecución se registran los campos:

- `status`: Estado de la ejecución (`OK`, `Error`)
- `subset`: subconjunto del corpus (C1, C2, C3, C4)
- `system`: Sistema que responde la consulta (LLM base, RAG, RLM)
- `model`: Modelo de cada sistema ('gpt-5.4-mini`, `gemini-2.5-flash`)
- `model_tag`: Etiqueta reducida del modelo de cada sistema (`gpt`, `gemini`) - `query_id`: Identificador de la consulta (ej. `Q01`)
- `level`: Nivel de complejidad de la consulta (extracción, comparación, síntesis)
- `rep`: Repetición de la ejecución (1, 2)
- `timestamp`: Fecha y hora de la ejecución
- `response`: Respuesta del sistema
- `latency_s`: Latencia en segundos
- `input_tokens`: Número de tokens de entrada
- `output_tokens`: Número de tokens de salida
- `cost_usd`: Coste de la ejecución en USD
- `judge_score`: Puntuación otorgada por el LLM juez

Adicionalmente, se incluyen campos específicos de cada sistema: `chunks` para RAG y `n_iterations` y `log_file` para RLM.

Las ejecuciones que superan el límite
de ventana de contexto de GPT-5.4 mini en C3 y C4 se registran con
`status=SKIPPED_CONTEXT_LIMIT`. Los errores no recuperables se registran con
`status=ERROR`.


In [ ]:
# Configuraciones de los modelos por sistema (modelo raíz + modelos sub-llamadas)
CONFIGS = [
    {'system': 'baseline', 'model': GPT_ROOT,    'tag': 'gpt'},
    {'system': 'baseline', 'model': GEMINI_ROOT, 'tag': 'gemini'},
    {'system': 'rag',      'model': GPT_ROOT,    'tag': 'gpt'},
    {'system': 'rag',      'model': GEMINI_ROOT, 'tag': 'gemini'},
    {'system': 'rlm',      'model': GPT_ROOT,    'tag': 'gpt',    'sub': GPT_SUB},
    {'system': 'rlm',      'model': GEMINI_ROOT, 'tag': 'gemini', 'sub': GEMINI_SUB},
]

# Bucle principal
for subset in SUBSETS:
    print(f'\n{"="*60}\nSUBCONJUNTO: {subset}\n{"="*60}')

    # Carga del subconjunto del corpus
    papers = load_subset(subset)
    corpus_text = corpus_to_prompt(papers)
    print(f'  {len(papers)} papers | {len(corpus_text):,} caracteres')

    for cfg in CONFIGS:
        system, model, tag = cfg['system'], cfg['model'], cfg['tag']

        # C3 y C4 se saltan para GPT-5.4 mini en LLM base
        if system == 'baseline' and tag == 'gpt' and subset in GPT_SKIP:
          print(f'  SKIP: {system}/{tag} en {subset} (limite de contexto)')
          for q in queries:
            for rep in range(1, N_REPS + 1):
              save_result(subset, system, tag, q['query_id'], rep, {
                'status': 'SKIPPED_CONTEXT_LIMIT',
                'subset': subset, 'system': system, 'model': model,
                'query_id': q['query_id'], 'rep': rep,
                'timestamp': datetime.now().isoformat()
              })
          continue

        print(f'\n  [{system}/{tag}]')

        # Preparar sistema
        rag_index = rag_tc = rlm_instance = None

        # Construcción del ínidce vectorial en RAG
        if system == 'rag':
            rag_index, rag_tc, embed_cost = build_rag_index(
                papers, model, f'rag_{subset}_{tag}')
            print(f'    Indice RAG listo (embedding: ${embed_cost:.4f})')

        # Bucle de consultas
        for q in queries:
            qid   = q['query_id']
            qtext = q['query']
            gt    = q['ground_truth']['answer']

            # Se consulta el checkpoint para comprobar si la consulta ha  sido ya
            # respondida
            for rep in range(1, N_REPS + 1):
                if already_done(subset, system, tag, qid, rep):
                    print(f'    SKIP checkpoint: {qid} rep{rep}')
                    continue

                print(f'    {qid} rep{rep}... ', end='', flush=True)

                # Respuesta del sistema
                try:
                    if system == 'baseline':
                      if tag == 'gpt':
                        res = run_baseline_gpt(corpus_text, qtext, model)
                      else:
                        res = run_baseline_gemini(corpus_text, qtext, model)
                    elif system == 'rag':
                        res = run_rag(rag_index, qtext, model, rag_tc)
                    elif system == 'rlm':
                      log_dir = os.path.join(RLM_LOGS_DIR, f'{subset}_{tag}')
                      rlm_instance = build_rlm(model, cfg['sub'], log_dir)
                      res = run_rlm(rlm_instance, corpus_text, qtext,
                              model, cfg['sub'], log_dir, qid, rep)

                    # Evaluación del LLM juez
                    judge = run_judge(qtext, gt, res['response'], q['level'])

                    # Registro de todos los campos
                    record = {
                        'status': 'OK',
                        'subset': subset, 'system': system,
                        'model': model, 'model_tag': tag,
                        'query_id': qid, 'level': q['level'], 'rep': rep,
                        'timestamp': datetime.now().isoformat(),
                        'response': res['response'],
                        'latency_s': res['latency_s'],
                        'input_tokens': res['input_tokens'],
                        'output_tokens': res['output_tokens'],
                        'cost_usd': res['cost_usd'],
                        'judge_score': judge['score'],
                        'judge_justification': judge['justification'],
                        'judge_cost_usd': judge['cost_usd'],
                        'total_cost_accumulated_usd': round(total_cost_usd, 4),
                    }
                    # Campos adicionales por sistema
                    if system == 'rag':
                        record['chunks'] = res.get('chunks', [])
                    if system == 'rlm':
                        record['n_iterations'] = res.get('n_iterations', 0)
                        record['log_file']     = res.get('log_file')

                    # Almacenar registro de resultado
                    save_result(subset, system, tag, qid, rep, record)
                    print(f'OK score={judge["score"]} | {res["latency_s"]}s | '
                          f'${res["cost_usd"]:.4f} | Total: ${total_cost_usd:.2f}')

                # Registro de errores
                except RuntimeError as e:
                    print(f'\nPRESUPUESTO AGOTADO: {e}')
                    raise
                except Exception as e:
                    print(f'ERROR: {e}')
                    save_result(subset, system, tag, qid, rep, {
                        'status': 'ERROR', 'error': str(e),
                        'subset': subset, 'system': system, 'model': model,
                        'query_id': qid, 'rep': rep,
                        'timestamp': datetime.now().isoformat()
                    })

print(f'\nExperimento completado. Coste total: ${total_cost_usd:.4f} USD')


## 10. Informe de resultados

Se guarda un informe CSV en el directorio `RESULTS_DIR` con los resultados de cada ejecución realizada en el bucle principal de experimentación. Adicionalmente, se muestran todas las ejecuciones válidas, el total de errores, y el número de ejecuciones saltadas, junto al coste total de la experimentación.


In [ ]:
# Informe de resultados
summary_path = os.path.join(RESULTS_DIR, 'experiment_summary.csv')
all_results = []
for f in sorted(Path(RESULTS_DIR).glob('*.json')):
    with open(f, encoding='utf-8') as jf:
        all_results.append(json.load(jf))

# Mostrar número de ejecuciones válidas, errores y ejecuciones saltadas
if all_results:
    keys = ['subset', 'system', 'model_tag', 'query_id', 'level', 'rep',
            'status', 'latency_s', 'cost_usd', 'judge_score', 'n_iterations']
    with open(summary_path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=keys, extrasaction='ignore')
        w.writeheader()
        w.writerows(all_results)

    ok      = sum(1 for r in all_results if r.get('status') == 'OK')
    errors  = sum(1 for r in all_results if r.get('status') == 'ERROR')
    skipped = sum(1 for r in all_results if 'SKIP' in r.get('status', ''))
    print(f'Total: {len(all_results)} | OK: {ok} | Errores: {errors} | Saltadas: {skipped}')
    print(f'Coste total: ${total_cost_usd:.4f} USD')
    print(f'Resumen guardado en: {summary_path}')
else:
    print('No hay resultados todavia')
